# 01 - RL Basics: Policy, Value, Return, Advantage

**Goal:** Understand the core RL quantities before treating PPO as a black box.

**What you will learn:** State, action, reward, return, value, Q, advantage, and the PPO clipping idea with tiny numeric examples.

**Inputs:** No trained agent is required; an optional short SoccerTwos rollout can provide real sparse rewards.

**Outputs:** Toy tables, value/advantage plots, and a PPO clipping visualization.

**Success criteria:** You can explain what a positive or negative advantage means for a PPO update.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

PROJECT_MARKER = Path("soccer_twos_project") / "notebook_tools.py"


def _running_in_colab():
    if "google.colab" in sys.modules:
        return True
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _candidate_project_roots():
    seen = set()

    def add(path):
        path = Path(path).expanduser()
        key = str(path)
        if key not in seen:
            seen.add(key)
            yield path

    for env_name in ("SOCCER_TWOS_PROJECT_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name)
        if value:
            yield from add(value)

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        yield from add(base)
        yield from add(base / "soccer-twos-starter")
        yield from add(base / "project" / "soccer-twos-starter")

    if sys.platform == "darwin":
        yield from add(
            Path.home()
            / "all_data"
            / "Georgia Tech"
            / "Course Content"
            / "CS 8803- DRL"
            / "project"
            / "soccer-twos-starter"
        )

    if _running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
        except Exception:
            pass
        for drive_root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"), Path("/content")):
            for relative in (
                Path("CS 8803- DRL") / "project" / "soccer-twos-starter",
                Path("project") / "soccer-twos-starter",
                Path("soccer-twos-starter"),
                Path("Colab Notebooks") / "soccer-twos-starter",
            ):
                yield from add(drive_root / relative)


def _find_project_root():
    for candidate in _candidate_project_roots():
        if (candidate / PROJECT_MARKER).exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find soccer_twos_project/notebook_tools.py. "
        "Open this notebook from the project root/notebooks folder, or set SOCCER_TWOS_PROJECT_ROOT."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for _module_name in list(sys.modules):
    if _module_name == "soccer_twos_project" or _module_name.startswith("soccer_twos_project."):
        del sys.modules[_module_name]

importlib.invalidate_caches()
from IPython.display import Markdown, display
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

LEARNING_DIR = learning_artifact_dir(ctx, "visual_learning")
print("Learning artifacts:", LEARNING_DIR)

## One Toy Trajectory

A trajectory is a short story: state, action, reward, next state. The policy chooses actions. The return is future reward added up with discounting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

toy = pd.DataFrame({
    "t": [0, 1, 2, 3],
    "state": ["far from ball", "near ball", "touch ball", "ball in goal"],
    "action": ["forward", "forward", "kick/move", "any"],
    "reward": [0.0, 0.0, 0.0, 1.0],
})
display(toy)

In [ ]:
rewards = toy["reward"].tolist()
gamma = 0.95
returns_table = value_advantage_table(rewards, gamma=gamma)
display(returns_table)
plot_value_advantage(returns_table, title="Discounted return from one delayed reward");

## Value, Q, And Advantage

`V(s)` predicts how good the state is before choosing an action. `Q(s, a)` predicts how good a particular action is. Advantage asks: was this action better or worse than expected?

In [ ]:
value_predictions = [0.10, 0.20, 0.35, 0.50]
q_predictions = [0.18, 0.30, 0.70, 0.80]
advantage_table = value_advantage_table(rewards, values=value_predictions, gamma=gamma)
advantage_table["q_estimate_Qsa"] = q_predictions
advantage_table["q_minus_value"] = advantage_table["q_estimate_Qsa"] - advantage_table["value_Vs"]
display(advantage_table)
plot_value_advantage(advantage_table, title="Return, value prediction, and advantage");

## Tiny Numeric Interpretation

If return is higher than value, advantage is positive, so PPO should make the chosen action more likely in similar states. If advantage is negative, PPO should make it less likely.

In [ ]:
worked = pd.DataFrame({
    "case": ["better than expected", "worse than expected"],
    "return_Gt": [1.00, 0.10],
    "value_Vs": [0.25, 0.40],
})
worked["advantage"] = worked["return_Gt"] - worked["value_Vs"]
worked["plain_english_update"] = [
    "increase probability of that action",
    "decrease probability of that action",
]
display(worked)

## PPO Clipping Intuition

PPO updates the policy, but clips very large probability-ratio changes. This keeps learning from making one huge unstable jump.

In [ ]:
ratios = np.linspace(0.5, 1.5, 200)
clip_eps = 0.2
clipped = np.clip(ratios, 1 - clip_eps, 1 + clip_eps)
positive_advantage_objective = np.minimum(ratios * 1.0, clipped * 1.0)
negative_advantage_objective = np.maximum(ratios * -1.0, clipped * -1.0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ratios, positive_advantage_objective, label="positive advantage")
ax.plot(ratios, negative_advantage_objective, label="negative advantage")
ax.axvline(1 - clip_eps, color="0.5", linestyle="--")
ax.axvline(1 + clip_eps, color="0.5", linestyle="--")
ax.set_title("PPO clipping limits how far one update can move")
ax.set_xlabel("new action probability / old action probability")
ax.set_ylabel("clipped surrogate objective")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()

## Real SoccerTwos Sparse Rewards

The next cell samples a short random SoccerTwos rollout. It stays headless by default, so it works on local Mac, Colab, and PACE when the environment is installed.

In [ ]:
RUN_SOCCER_SAMPLE = True
soccer_rollout = None

if RUN_SOCCER_SAMPLE:
    try:
        soccer_rollout = collect_single_player_rollout(
            policy="random",
            steps=150,
            render=False,
            label="random SoccerTwos sample",
        )
        display(rollout_summary_table({"random soccer": soccer_rollout}))
        plot_reward_timeline(soccer_rollout, title="Sparse rewards in a random SoccerTwos rollout");
    except Exception as exc:
        print("SoccerTwos rollout skipped:", type(exc).__name__, exc)
else:
    print("Set RUN_SOCCER_SAMPLE=True to sample real SoccerTwos rewards.")

In [ ]:
if soccer_rollout is not None and not soccer_rollout.empty:
    recent_rewards = soccer_rollout["reward"].tail(25).tolist()
    soccer_value_table = value_advantage_table(recent_rewards, gamma=0.99)
    display(soccer_value_table.tail(10))
    plot_value_advantage(soccer_value_table, title="Returns from a short SoccerTwos reward window");

## Why PPO Fits This Project

SoccerTwos uses continuous-looking movement but discrete branched actions. PPO is a practical policy-gradient method for this setting because it learns directly from rollout batches, supports neural policies, and limits update size with clipping.

## Key Takeaways

Return is what happened after a state; value is what the network expected; advantage is the difference that tells PPO how to adjust action probabilities. PPO is useful here because it can learn a neural movement policy from batches of SoccerTwos rollouts.

## What To Run Next

Run `02_tiny_ppo_training_watch.ipynb` to watch PPO produce logs, checkpoints, and behavior changes.